In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

RANGES = {'1500-2000 cm-1 (Amide I)': (1500, 2000), '2800-3000 cm-1 (CH stretch)': (2800, 3000)}
RANGE_COLORS = {'1500-2000 cm-1 (Amide I)': '#3070b0', '2800-3000 cm-1 (CH stretch)': '#e08030'}
DATA_DIR = '../../data'

## 1. Load Aligned Data

In [ ]:
adult = pd.read_excel(f'{DATA_DIR}/raman_functional_intensity_matrix_bspline_n4_aligned_402_3499.xlsx',
                       sheet_name='Aligned Spectra', index_col=0)
valid = pd.read_excel(f'{DATA_DIR}/raman_valid_samples_combined_aligned_402_3499.xlsx',
                       sheet_name='Aligned Spectra', index_col=0)
cons  = pd.read_excel(f'{DATA_DIR}/raman_consensus_spectra_aligned_402_3499.xlsx',
                       sheet_name='Aligned Spectra', index_col=0)
evalset = pd.read_excel(f'{DATA_DIR}/Eval_Preprocessed_aligned_402_3499_recalibrated.xlsx',
                         sheet_name='Aligned Spectra', index_col=0)

x = adult.index.values.astype(float)
assert (adult.index == valid.index).all() and (adult.index == cons.index).all() and (adult.index == evalset.index).all(), 'grids must match'
mean_ref = cons['Arithmetic Mean']

print(f'Adulterant-panel dataset   : {adult.shape[1]} samples x {adult.shape[0]} wavenumber points')
print(f'Quality-controlled dataset : {valid.shape[1]} samples x {valid.shape[0]} wavenumber points')
print(f'Evaluation dataset (held out): {evalset.shape[1]} samples x {evalset.shape[0]} wavenumber points')
print(f'Common grid                : {x.min():.0f} - {x.max():.0f} cm-1')

## 2. Ground-Truth Labels

In [ ]:
ADULTERANT_KEYWORDS = ['doxy', 'enro', 'urea', 'water', 'starch', 'veg oil', 'amoxicillin']

def is_adulterated(name):
    n = name.lower()
    return any(k in n for k in ADULTERANT_KEYWORDS)

labels_panel = pd.Series({c: int(is_adulterated(c)) for c in adult.columns}, name='Adulterated')
labels_valid = pd.Series(0, index=valid.columns, name='Adulterated')  # QC dataset is all confirmed genuine

dev = pd.concat([adult, valid], axis=1)
dev_labels = pd.concat([labels_panel, labels_valid]).loc[dev.columns]

print(f'Adulterant-panel file : {labels_panel.sum()} adulterated / {(labels_panel == 0).sum()} genuine controls')
print(f'Quality-controlled file: {len(labels_valid)} genuine (all)')
print(f'Development set: {dev.shape[1]} samples -> {dev_labels.sum()} adulterated / {(dev_labels == 0).sum()} genuine')

# Evaluation-set ground truth: treatment keyword in filename
EVAL_ADULTERANT_KEYWORDS = ['starch', 'urea', 'dilution', 'vegetable oil']

def is_adulterated_eval(name):
    n = name.lower()
    return any(k in n for k in EVAL_ADULTERANT_KEYWORDS)

eval_labels = pd.Series({c: int(is_adulterated_eval(c)) for c in evalset.columns}, name='Adulterated').loc[evalset.columns]
print()
print(f'Evaluation set: {evalset.shape[1]} samples -> {eval_labels.sum()} adulterated / {(eval_labels == 0).sum()} genuine')
print(eval_labels)

## 3. Zoomed-Window Visual Comparison — Development Samples vs Arithmetic Mean

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (rname, (lo, hi)) in zip(axes, RANGES.items()):
    mask = (x >= lo) & (x <= hi)
    for col in dev.columns:
        color = '#d62728' if dev_labels[col] else '#2ca02c'
        ax.plot(x[mask], dev[col].values[mask], color=color, alpha=0.2, linewidth=0.6)
    ax.plot(x[mask], mean_ref.values[mask], color='black', linewidth=2, label='Arithmetic Mean')
    ax.set_title(rname)
    ax.set_xlabel('Raman Shift (cm-1)')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Normalised Intensity (a.u.)')
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='#d62728', label='Adulterated sample'),
           Line2D([0], [0], color='#2ca02c', label='Genuine sample')]
fig.legend(handles=handles, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.06), fontsize=9)
plt.tight_layout()
plt.show()

## 4. L2 Distance to Arithmetic Mean, per Zoom Window — Development Set

In [ ]:
def region_l2(df, ref, lo, hi):
    mask = (x >= lo) & (x <= hi)
    r = ref.values[mask]
    return pd.Series(np.linalg.norm(df.values[mask, :] - r[:, None], axis=0), index=df.columns)

dev_l2 = pd.DataFrame({rname: region_l2(dev, mean_ref, lo, hi) for rname, (lo, hi) in RANGES.items()})
dev_l2.round(4).head()

## 5. Youden's J Threshold Selection

For each zoom window, `roc_curve` sweeps every candidate threshold on the development-set L2 scores; the threshold maximising `J = TPR - FPR` (equivalently Sensitivity + Specificity - 1) is kept. A **combined** score is also built by z-scoring each window's L2 distance against the quality-controlled population's own mean/SD in that window (so the two windows are on a comparable scale) and taking the max across windows; its own Youden's-J-optimal threshold is found the same way.

In [ ]:
def youden_threshold(scores, y_true):
    fpr, tpr, thr = roc_curve(y_true, scores)
    j = tpr - fpr
    i = int(np.argmax(j))
    return dict(threshold=thr[i], sensitivity=tpr[i], specificity=1 - fpr[i], j=j[i])

# Per-region z-scaling stats, from the quality-controlled (genuine) population only —
# used only to put the two windows on a common scale for the combined score.
qc_stats = {}
for rname, (lo, hi) in RANGES.items():
    l2 = region_l2(valid, mean_ref, lo, hi)
    qc_stats[rname] = dict(mean=l2.mean(), std=l2.std(ddof=1))

dev_z = pd.DataFrame({rname: (dev_l2[rname] - qc_stats[rname]['mean']) / qc_stats[rname]['std']
                      for rname in RANGES})
dev_combo_score = dev_z.max(axis=1)

youden = {}
for rname in RANGES:
    youden[rname] = youden_threshold(dev_l2[rname].values, dev_labels.values)
youden['Combined (max z-score across windows)'] = youden_threshold(dev_combo_score.values, dev_labels.values)

youden_df = pd.DataFrame(youden).T
youden_df.round(4)

## 6. Development-Set Classification Report at the Youden's-J Threshold

In [ ]:
def classify_report(name, scores, threshold, y_true):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    acc = (tp + tn) / (tp + tn + fp + fn)
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    auc = roc_auc_score(y_true, scores)
    return dict(Method=name, Threshold=threshold, TP=tp, FP=fp, TN=tn, FN=fn,
                Accuracy=acc, Sensitivity=sens, Specificity=spec, AUC=auc)

dev_rows = []
for rname in RANGES:
    dev_rows.append(classify_report(rname, dev_l2[rname].values, youden_df.loc[rname, 'threshold'], dev_labels.values))
dev_rows.append(classify_report('Combined (max z-score across windows)', dev_combo_score.values,
                                 youden_df.loc['Combined (max z-score across windows)', 'threshold'], dev_labels.values))

dev_results_df = pd.DataFrame(dev_rows).set_index('Method')
dev_results_df.round(4)

## 7. ROC Curves — Development Set, with Youden's-J Operating Point Marked

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for rname in RANGES:
    fpr, tpr, _ = roc_curve(dev_labels.values, dev_l2[rname].values)
    auc = roc_auc_score(dev_labels.values, dev_l2[rname].values)
    ax.plot(fpr, tpr, linewidth=1.5, color=RANGE_COLORS[rname], label=f'{rname} (AUC={auc:.3f})')
    y = youden_df.loc[rname]
    ax.scatter([1 - y['specificity']], [y['sensitivity']], color=RANGE_COLORS[rname], zorder=5, s=50,
               edgecolor='black', label=f'  Youden J* (Sens={y["sensitivity"]:.2f}, Spec={y["specificity"]:.2f})')

fpr, tpr, _ = roc_curve(dev_labels.values, dev_combo_score.values)
auc = roc_auc_score(dev_labels.values, dev_combo_score.values)
ax.plot(fpr, tpr, linewidth=1.8, color='black', linestyle='-.', label=f'Combined (AUC={auc:.3f})')
y = youden_df.loc['Combined (max z-score across windows)']
ax.scatter([1 - y['specificity']], [y['sensitivity']], color='black', zorder=5, s=60,
           edgecolor='white', label=f'  Youden J* (Sens={y["sensitivity"]:.2f}, Spec={y["specificity"]:.2f})')

ax.plot([0, 1], [0, 1], color='grey', linestyle='--', linewidth=1, label='Chance')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC — Development Set (Adulterated vs Genuine)')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

## 8. Apply the Calibrated Pipeline to the Held-Out Evaluation Dataset

The Arithmetic Mean reference, the two zoom windows, the QC-population z-scaling stats, and the three Youden's-J thresholds are all fixed from Section 5 — nothing is re-fit here. This section only computes L2 distances for the 12 evaluation samples and applies the existing rule.

**Recalibrated input:** the evaluation set used here is the per-wavenumber batch-standardised (recalibrated) file from `raman_final_dataset_alignment.ipynb`, correcting the cross-setup batch offset before the pipeline is applied.

In [ ]:
eval_l2 = pd.DataFrame({rname: region_l2(evalset, mean_ref, lo, hi) for rname, (lo, hi) in RANGES.items()})
eval_z = pd.DataFrame({rname: (eval_l2[rname] - qc_stats[rname]['mean']) / qc_stats[rname]['std']
                       for rname in RANGES})
eval_combo_score = eval_z.max(axis=1)

eval_rows = []
for rname in RANGES:
    eval_rows.append(classify_report(rname, eval_l2[rname].values, youden_df.loc[rname, 'threshold'], eval_labels.values))
eval_rows.append(classify_report('Combined (max z-score across windows)', eval_combo_score.values,
                                  youden_df.loc['Combined (max z-score across windows)', 'threshold'], eval_labels.values))

eval_results_df = pd.DataFrame(eval_rows).set_index('Method')
eval_results_df.round(4)

## 9. Per-Sample Evaluation-Set Scores & Predictions

In [ ]:
eval_detail = eval_l2.copy()
eval_detail.columns = [f'{r} L2 distance' for r in eval_detail.columns]
eval_detail.insert(0, 'Adulterated (ground truth)', eval_labels)
for rname in RANGES:
    eval_detail[f'{rname} predicted'] = (eval_l2[rname].values >= youden_df.loc[rname, 'threshold']).astype(int)
eval_detail['Combined score (max z)'] = eval_combo_score
eval_detail['Combined predicted'] = (eval_combo_score.values >= youden_df.loc['Combined (max z-score across windows)', 'threshold']).astype(int)
eval_detail.round(4)

## 10. Save Results

In [ ]:
OUT_PATH = 'adulterated_classification_results.xlsx'

dev_scores_out = dev_l2.copy()
dev_scores_out.columns = [f'{r} L2 distance' for r in dev_scores_out.columns]
dev_scores_out.insert(0, 'Adulterated (ground truth)', dev_labels)
dev_scores_out['Source file'] = ['adulterant-panel' if c in adult.columns else 'quality-controlled' for c in dev_scores_out.index]
dev_scores_out['Combined score (max z)'] = dev_combo_score

with pd.ExcelWriter(OUT_PATH) as writer:
    dev_results_df.to_excel(writer, sheet_name='Dev Set Classification')
    eval_results_df.to_excel(writer, sheet_name='Eval Set Classification')
    youden_df.to_excel(writer, sheet_name='Youden Thresholds')
    dev_scores_out.to_excel(writer, sheet_name='Dev Set Per-Sample Scores')
    eval_detail.to_excel(writer, sheet_name='Eval Set Per-Sample Scores')

print(f'Saved -> {OUT_PATH}')
print()
print('=== Development set (threshold calibration) ===')
print(dev_results_df[['AUC', 'Accuracy', 'Sensitivity', 'Specificity']].round(3))
print()
print('=== Evaluation set (held-out generalisation test) ===')
print(eval_results_df[['AUC', 'Accuracy', 'Sensitivity', 'Specificity']].round(3))

## 11. Bootstrap 95% CI for reported AUCs

In [ ]:
# Bootstrap 95% CI for the reported ROC AUCs (stratified resampling of the fixed scores and labels)
def auc_ci(y_true, y_score, n_boot=5000, alpha=0.05, seed=0):
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    rb = np.random.default_rng(seed)
    i0, i1 = np.flatnonzero(y_true == 0), np.flatnonzero(y_true == 1)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        ii = np.concatenate([rb.choice(i0, i0.size, replace=True),
                             rb.choice(i1, i1.size, replace=True)])
        boot[b] = roc_auc_score(y_true[ii], y_score[ii])
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return roc_auc_score(y_true, y_score), lo, hi

def auc_ci_table(labels, l2_df, combo_score):
    rows = {r: auc_ci(labels, l2_df[r].values) for r in RANGES}
    rows['Combined (max z-score across windows)'] = auc_ci(labels, combo_score.values)
    return pd.DataFrame(rows, index=['AUC', 'CI 2.5%', 'CI 97.5%']).T

dev_auc_ci_df  = auc_ci_table(dev_labels.values,  dev_l2,  dev_combo_score)
eval_auc_ci_df = auc_ci_table(eval_labels.values, eval_l2, eval_combo_score)
print('=== Development set - AUC 95% CI ===')
print(dev_auc_ci_df.round(3))
print()
print('=== Evaluation set - AUC 95% CI ===')
print(eval_auc_ci_df.round(3))